In [39]:
import glob
import copy
import numpy as np

# CityLearn Imports
from citylearn.citylearn import CityLearnEnv
from citylearn.agents.rbc import OptimizedRBC as Agent

# Train (no need to, so definition of RBC here)

In [41]:
# ------------------------------------------------------------------
# 0. Global Parameters
# ------------------------------------------------------------------
num_houses = 1
base_schema = "citylearn_challenge_2023_phase_3_1"

In [42]:
temp_env = CityLearnEnv(schema=base_schema)
schema_dict = copy.deepcopy(temp_env.schema)
building_names = list(schema_dict["buildings"].keys())

if num_houses < len(building_names):
    target_buildings = building_names[:num_houses]
    schema_dict["buildings"] = {b: schema_dict["buildings"][b] for b in target_buildings}

INFO:root:Go here /home/ntpt/.cache/citylearn/v2.5.0/datasets/citylearn_challenge_2023_phase_3_1/schema.json 


In [23]:
class CentralizedRBCAgent:
    def __init__(self, env):
        self.action_space = env.action_space
        # Store the legend to map the flat array back into readable variables
        self.obs_names = env.observation_names
        self.building_names = [b.name for b in env.buildings]

    def predict(self, observation):
        """
        Dynamically parses the flat centralized array and applies RBC heuristics.
        """
        # 1. Unpack the nested list and map it to a dictionary
        flat_obs = observation[0]
        state = dict(zip(self.obs_names, flat_obs))

        # Extract global variables (These don't have building prefixes)
        hour = state.get("hour", 12)

        centralized_action = []

        # 2. Iterate through each building to apply local heuristics
        for b_name in self.building_names:
            # Initialize neutral actions: [DHW, Battery, Cooling]
            building_action = [0.0, 0.0, 0.0]

            # Dynamically extract this specific building's local telemetry
            indoor_temp = state.get(f"{b_name}-indoor_dry_bulb_temperature")
            setpoint = state.get(f"{b_name}-indoor_dry_bulb_temperature_set_point")

            # --- Battery Heuristics ---
            if hour >= 17 and hour <= 21:
                building_action[1] = -0.5  # Discharge
            elif hour >= 2 and hour <= 6:
                building_action[1] = 0.5  # Charge

            # --- HVAC Heuristics ---
            # Fallback to 0.0 if the environment doesn't provide temperature/setpoint data
            if indoor_temp is not None and setpoint is not None:
                if indoor_temp > (setpoint + 1.0):
                    # Actuate cooling if temperature exceeds upper deadband
                    building_action[2] = min(0.2 * (indoor_temp - setpoint), 1.0)

            # Append to the master centralized array
            centralized_action.extend(building_action)

        # 3. Clip and return the final action array
        action_array = np.array([centralized_action], dtype=np.float32)
        return np.clip(action_array, self.action_space.low, self.action_space.high)

# Evaluate (get KPIs)

In [43]:
# initialize
env = CityLearnEnv(schema_dict, central_agent=True)
model = Agent(env)

# step through environment and apply agent actions
observations, _ = env.reset()

while not env.terminated:
    actions = model.predict(observations)
    observations, reward, info, terminated, truncated = env.step(actions)

In [44]:
# test
kpis = model.env.evaluate()
kpis = kpis.pivot(index="cost_function", columns="name", values="value").round(3)
kpis = kpis.dropna(how="all")
display(kpis)

name,Building_1,District
cost_function,,
all_time_peak_average,NaN,1.138
annual_normalized_unserved_energy_total,0.016,0.016
carbon_emissions_total,2.141,2.141
cost_total,2.013,2.013
daily_one_minus_load_factor_average,NaN,0.782
daily_peak_average,NaN,1.373
discomfort_cold_delta_average,9.362,9.362
discomfort_cold_delta_maximum,14.991,14.991
discomfort_cold_delta_minimum,0.000,0.000


In [33]:
env.action_names

[['dhw_storage',
  'electrical_storage',
  'cooling_device',
  'dhw_storage',
  'electrical_storage',
  'cooling_device']]

In [34]:
env.action_space

[Box([-1. -1.  0. -1. -1.  0.], 1.0, (6,), float32)]

In [35]:
env.observation_space

[Box([ 1.0000000e+00  1.0000000e+00  1.8160000e+01  1.7735878e+01
   1.8267269e+01  1.7025522e+01  0.0000000e+00  0.0000000e+00
   0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
   0.0000000e+00  0.0000000e+00  2.7006584e-01  7.6293945e-06
   3.0811283e-01  0.0000000e+00  0.0000000e+00  0.0000000e+00
  -4.6562166e+00  3.0250000e-02  3.0250000e-02  3.0250000e-02
   3.0250000e-02  0.0000000e+00  0.0000000e+00  0.0000000e+00
   0.0000000e+00  2.0000000e+01  1.4687138e+00  1.6775879e-01
   0.0000000e+00  0.0000000e+00  0.0000000e+00 -3.6486399e+00
   0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
   2.1666666e+01], [7.0000000e+00 2.4000000e+01 3.8110001e+01 3.8912338e+01 3.8959305e+01
  3.9400982e+01 5.0084000e+02 5.6977252e+02 5.7735968e+02 6.3289221e+02
  9.1862000e+02 1.0693932e+03 1.1619512e+03 1.1784778e+03 5.6347221e-01
  4.7222275e+01 7.4589686e+00 1.7110062e+00 1.0000000e+00 1.0000000e+00
  1.9749760e+01 6.6050000e-02 6.6050000e-02 6.6050000e-02 6.6050000e-0

In [38]:
env.observation_names

[['day_type',
  'hour',
  'outdoor_dry_bulb_temperature',
  'outdoor_dry_bulb_temperature_predicted_1',
  'outdoor_dry_bulb_temperature_predicted_2',
  'outdoor_dry_bulb_temperature_predicted_3',
  'diffuse_solar_irradiance',
  'diffuse_solar_irradiance_predicted_1',
  'diffuse_solar_irradiance_predicted_2',
  'diffuse_solar_irradiance_predicted_3',
  'direct_solar_irradiance',
  'direct_solar_irradiance_predicted_1',
  'direct_solar_irradiance_predicted_2',
  'direct_solar_irradiance_predicted_3',
  'carbon_intensity',
  'indoor_dry_bulb_temperature',
  'non_shiftable_load',
  'solar_generation',
  'dhw_storage_soc',
  'electrical_storage_soc',
  'net_electricity_consumption',
  'electricity_pricing',
  'electricity_pricing_predicted_1',
  'electricity_pricing_predicted_2',
  'electricity_pricing_predicted_3',
  'cooling_demand',
  'dhw_demand',
  'occupant_count',
  'power_outage',
  'indoor_dry_bulb_temperature_cooling_set_point',
  'indoor_dry_bulb_temperature',
  'non_shiftable_lo